# 강의 05 · 실습 2 — 평가 데이터셋과 채점자 · (2-2) 빈칸 채우기 II


## 1. 문제상황

- 놀이공원 구름월드의 안내 서비스는 FAQ 문서를 근거로 손님의 질문에 답합니다.
- 운영 담당자는 서비스가 낸 답이 FAQ 내용과 맞는지 답마다 눈으로 읽어 확인합니다.
- 프롬프트나 모델을 바꿀 때마다 담당자는 같은 질문을 다시 넣고 답을 다시 읽어야 합니다.
- 답이 좋아졌는지 나빠졌는지를 담당자는 숫자로 말할 수 없고, 확인한 기록도 남지 않습니다.


## 2. 문제와 목표

- **문제**: 답의 품질 확인이 담당자의 눈에 의존하고, 확인 결과가 숫자와 기록으로 남지 않습니다.
- **목표**
  - 질문과 기대 조건을 짝지은 데이터셋을 등록합니다.
    - 데이터셋 `sesac-lec05-ex02-golden`: 질문 4개(인사, 환불 규정, 야간 퍼레이드, FAQ 밖의 파이썬 질문)와 「어떻게 답해야 한다」는 기대 조건 문장의 짝
  - 규칙 채점자와 모델 채점자로 서비스의 답을 자동 채점하고, 실험 기록을 LangSmith에 남깁니다.
    - 규칙 채점자 `rule_content`: FAQ 밖 질문이면 답에 「확인할 수 없」 표현이 있는지, FAQ 안 질문이면 답이 비어 있지 않은지로 0 또는 1점
    - 모델 채점자 `judge_faithful`: 구조화 출력을 붙인 모델이 답이 기대 조건에 부합하고 지어낸 내용이 없는지 판정해 0 또는 1점
  - 모델 채점자 점수의 평균이 기준에 못 미치면 차단으로 판정하는 게이트를 둡니다.
    - 기준: 0.5
- **목표 달성 여부의 판정 기준**:
  - 데이터셋의 질문 4개마다 규칙 점수와 모델 점수가 화면에 출력되고,
  - 모델 점수의 평균이 기준 0.5와 비교되어 통과 또는 차단이 마지막 줄에 출력되는 것을 확인합니다.
  - LangSmith 화면의 데이터셋 `sesac-lec05-ex02-golden`에 실험이 남는 것을 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex02_s1_diagram.svg)


## 4. 단계별 요구사항

1. **골든 데이터셋을 등록합니다.**
    - 질문 4개와 기대 조건(`expect`)·FAQ 안 질문 여부(`must_know`)를 짝지어 `sesac-lec05-ex02-golden` 데이터셋으로 올립니다.
    - 같은 이름의 데이터셋이 이미 있으면 새로 만들지 않고 재사용합니다.
2. **평가 대상을 지정합니다.**
    - `target` 함수는 `inputs` 딕셔너리에서 질문을 꺼내 안내 서비스 `answer`를 부르고, 답을 `{"answer": 답}` 딕셔너리로 돌려줍니다.
3. **규칙 채점자를 만듭니다.**
    - `rule_content`는 FAQ 밖 질문이면 답에 「확인할 수 없」 표현이 있는지 보고, FAQ 안 질문이면 답이 비어 있지 않은지 보고, 0 또는 1점을 `{"key": "rule_content", "score": 점수}`로 돌려줍니다.
4. **모델 채점자를 만듭니다.**
    - `Judge` 스키마(`faithful`·`reason`)를 선언하고, 구조화 출력을 붙인 모델이 질문·기대 조건·답을 읽고 부합 여부를 판정해 `{"key": "judge_faithful", "score": 점수, "comment": 근거}`로 돌려줍니다.
5. **실험을 실행합니다.**
    - `client.evaluate`에 평가 대상·데이터셋 이름·채점자 목록을 넣어 실행하고, 질문마다 규칙 점수와 모델 점수를 표로 출력합니다.
    - `client.evaluate`가 화면에 출력하는 실험 URL 줄은 `redirect_stdout`으로 잡아 두고, 대신 「실험 URL: (LangSmith 화면에서 확인)」 한 줄을 출력합니다.
6. **게이트를 판정합니다.**
    - 모델 채점자 점수의 평균을 기준 0.5와 비교해 통과 또는 차단을 출력합니다.


## 5. 코드 골격 — LangSmith 평가 4단

LangSmith로 평가 절차를 세우는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 데이터셋 등록 | 질문과 기대 조건을 짝지어 데이터셋으로 올립니다 | `client.create_dataset(...)`, `client.create_examples(...)` | 1 |
| ② 평가 대상 지정 | 입력을 받아 답을 돌려주는 함수 하나를 평가 대상으로 지정합니다 | `def target(inputs: dict) -> dict` | 2 |
| ③ evaluator 정의 | 채점 함수를 만듭니다. 규칙 채점과 모델 채점 두 벌을 씁니다 | `def rule_content(...)`, `with_structured_output(Judge)` | 3, 4 |
| ④ 실행·게이트 | 셋을 넣어 실험을 돌리고, 평균 점수를 기준과 비교해 통과와 차단을 정합니다 | `client.evaluate(target, data=..., evaluators=[...])` | 5, 6 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고, 키를 읽고, 추적 설정을 켜고, 모델을 준비합니다. `warnings.filterwarnings` 두 줄은 추적 라이브러리가 내는 직렬화 경고와 진행 막대 경고를 화면에서 감춥니다. 동작에는 영향이 없습니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- `LANGSMITH_TRACING`을 켜고 프로젝트 이름을 `sesac-lec05-ex02`로 정하면, 아래에서 부르는 모델 호출과 실험이 LangSmith의 그 프로젝트에 남습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [ ]:
# 여기에 단계 0(라이브러리 불러오기, 경고 억제, .env 읽기, 추적 설정, 모델 준비)을 작성합니다.

평가 대상이 될 안내 서비스입니다. 이 서비스는 이미 만들어 둔 부품이며, 이 실습에서는 고치지 않습니다. 함수 `answer`가 질문을 받아 FAQ만 근거로 답을 돌려줍니다.


In [ ]:
FAQ = """
[환불] Q: 자유이용권 환불 규정 알려 주세요
A: 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다.
[운영] Q: 운영 시간이 어떻게 되나요?
A: 평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다.
[야간] Q: 야간개장은 언제 하나요?
A: 금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다.
[주차] Q: 주차 요금은 얼마인가요?
A: 자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다.
"""

SERVICE_GUIDE = (
    "너는 놀이공원 구름월드의 안내 담당자다. 아래 FAQ에 적힌 내용만 근거로 두 문장 안에서 답한다. "
    "인사말에는 짧은 인사로 답한다. "
    "FAQ에 없는 내용을 물으면 '해당 내용은 확인할 수 없습니다.'라고만 답한다.\n"
    "=== FAQ ===\n" + FAQ
)


def answer(question: str) -> str:
    """안내 서비스: 질문을 받아 FAQ만 근거로 답을 돌려준다."""
    res = llm.invoke([("system", SERVICE_GUIDE), ("human", question)])
    return res.content.strip()


print(answer("자유이용권 환불 규정 알려 주세요"))

### 단계 ① — 데이터셋 등록 (요구사항 1)

LangSmith의 데이터셋은 예시의 모음이고, 예시 하나는 `inputs`(질문)와 `outputs`(기준)의 짝입니다. 채점자는 나중에 이 `outputs`를 `reference_outputs`라는 이름으로 받습니다. 같은 이름의 데이터셋을 다시 만들면 오류가 나므로, 먼저 `has_dataset`으로 있는지 확인하고 있으면 재사용합니다.


In [ ]:
# 여기에 단계 ①(데이터셋 등록 — 이미 있으면 재사용)을 작성합니다.

### 단계 ② — 평가 대상 지정 (요구사항 2)

평가 대상은 `inputs` 딕셔너리를 받아 딕셔너리를 돌려주는 함수 하나입니다. LangSmith가 데이터셋의 질문마다 이 함수를 부르고, 돌려받은 딕셔너리를 채점자의 `outputs`로 넘깁니다. 안내 서비스 자체는 고치지 않고, 서비스를 감싸는 함수만 씁니다.


In [ ]:
# 여기에 단계 ②(평가 대상 함수 target 정의)를 작성합니다.

### 단계 ③ — evaluator 정의 (요구사항 3, 4)

- 채점자는 `inputs`·`outputs`·`reference_outputs` 세 딕셔너리를 받아 `{"key": 이름, "score": 점수}` 딕셔너리를 돌려주는 함수입니다.
- 규칙 채점자 `rule_content`는 모델을 부르지 않고 문자열 검사만으로 점수를 냅니다.
- 모델 채점자 `judge_faithful`은 `Judge` 스키마를 강제한 모델을 한 번 불러 `faithful` 값을 점수로, `reason` 값을 근거(`comment`)로 돌려줍니다.


In [ ]:
# 여기에 단계 ③(규칙 채점자 rule_content, Judge 스키마, 모델 채점자 judge_faithful 정의)을 작성합니다.

### 단계 ④ — 실행·게이트 (요구사항 5, 6)

`client.evaluate`가 데이터셋의 질문마다 평가 대상을 부르고, 세 쌍을 채점자마다 넘겨 점수를 모읍니다. 결과를 돌면서 질문마다 두 점수를 표로 출력하고, 모델 채점자 점수의 평균을 기준과 비교해 통과 또는 차단을 정합니다. `max_concurrency=1`은 질문을 하나씩 차례대로 평가하라는 뜻입니다. `client.evaluate`는 실험 URL을 화면에 직접 출력하므로, `redirect_stdout`으로 그 출력을 잡아 두고 URL 대신 안내 문장을 출력합니다.


In [ ]:
# 여기에 단계 ④(redirect_stdout으로 감싼 client.evaluate 실행, 점수 표 출력, 게이트 판정)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 네 가지를 확인합니다.

1. 단계 ① 출력에 데이터셋 이름 `sesac-lec05-ex02-golden`과 「만들고 예시 4개를 올렸습니다」 또는 「이미 있어 재사용합니다」 문장이 출력됩니다.
2. 단계 ③ 출력에서 FAQ 밖 질문(파이썬 리스트 정렬)의 규칙 점수가 1입니다. 답에 「확인할 수 없습니다」가 들어 있기 때문입니다. 모델 채점자 출력에는 `score`와 `comment`가 함께 출력됩니다.
3. 단계 ④ 표에 질문 4개가 한 줄씩 출력되고, 규칙 점수와 모델 점수가 각각 0 또는 1입니다.
4. 단계 ④ 첫 줄에 실험 이름이, 둘째 줄에 「실험 URL: (LangSmith 화면에서 확인)」이 출력되고, 마지막 줄에 모델 점수 평균과 기준 0.5, 통과 또는 차단이 출력됩니다. LangSmith 화면의 데이터셋 `sesac-lec05-ex02-golden`에 `sesac-v1`로 시작하는 실험이 새로 생깁니다.

네 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다.
